# Lab 2: Llamadas al LLM y Construcción de Prompts Aumentados

Bienvenido a **Llamadas al LLM y Construcción de Prompts Aumentados**.
En este lab aprenderás a usar dos funciones esenciales para interactuar
con Modelos de Lenguaje (LLMs): una para preguntas simples y otra para
conversaciones. El objetivo principal es mostrarte cómo agregar información
extra a tus prompts para que el modelo dé respuestas más precisas y útiles.

## En este lab aprenderás:
- Cómo enviar preguntas simples y conversaciones a un LLM
- Cómo usar datos adicionales para enriquecer tus prompts y mejorar las respuestas

---

## Tabla de Contenidos
- [1 - Entendiendo las funciones para llamar al LLM](#1)
    - [1.1 `generate_with_single_input` — pregunta simple](#1-1)
    - [1.2 `generate_with_multiple_input` — conversación múltiple](#1-2)
- [2 - Integrando datos en un prompt](#2)
    - [2.1 Entendiendo la estructura de datos](#2-1)
    - [2.2 Creando el prompt aumentado](#2-2)

<a id='1'></a>
## 1 - Entendiendo las funciones para llamar al LLM

En esta sección explorarás las funciones principales que usaremos para llamar
a los LLMs durante todo el curso. Nosotros usamos la API de OpenAI con
el modelo `gpt-4o-mini`, que es equivalente al Qwen que usa el curso en Coursera.

<a id='1-1'></a>
### 1.1 `generate_with_single_input` — Pregunta simple

Esta función te permite generar texto a partir de un único prompt de entrada.
Cada llamada es **independiente** — el modelo no recuerda nada de llamadas anteriores.

#### Parámetros:
- `prompt` (str): El texto que le envías al modelo
- `max_tokens` (int): Máximo de tokens (palabras) que puede responder. Default: `1024`
- `model` (str): El modelo a usar. Default: `"gpt-4o-mini"`

#### ¿Qué devuelve?
Un diccionario con dos claves:
- `role` → siempre será `"assistant"`
- `content` → el texto de la respuesta del modelo

In [15]:
# Importamos las funciones desde utils.py
import sys
sys.path.append('../../')

from utils import generate_with_single_input, generate_with_multiple_input

In [16]:
# Ejemplo de llamada simple — igual que el curso
output = generate_with_single_input(
    prompt="¿Cuál es la capital de Francia?"
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: La capital de Francia es París.


<a id='1-2'></a>
### 1.2 `generate_with_multiple_input` — Conversación múltiple

Esta función maneja múltiples mensajes en un contexto conversacional.
El formato de entrada es una lista de diccionarios, cada uno con dos claves:

1. `role` — quién habla: puede ser `user`, `assistant` o `system`
2. `content` — el texto del mensaje

Esto permite simular una conversación completa con historial,
donde el modelo "recuerda" todo lo que se dijo antes porque
se lo estamos enviando nosotros explícitamente en cada llamada.

In [17]:
# Conversación con historial — el modelo necesita el contexto anterior
# para poder responder la última pregunta correctamente
messages = [
    {'role': 'user', 'content': '¿Quién ganó el Mundial de Fútbol en 2018?'},
    {'role': 'assistant', 'content': 'Francia ganó el Mundial de Fútbol 2018.'},
    {'role': 'user', 'content': '¿Quién era el capitán?'}
]

output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: El capitán de la selección francesa durante el Mundial de Fútbol 2018 fue Hugo Lloris, quien es el portero del equipo.


<a id='2'></a>
## 2 - Integrando datos en un prompt

En esta sección aprenderás a incorporar datos en un prompt antes de
pasárselo al LLM. Trabajaremos con un pequeño dataset de casas para
entender cómo se aumentan los prompts en el contexto de RAG.

<a id='2-1'></a>
### 2.1 Entendiendo la estructura de datos

Primero veamos la estructura de los datos. Es un dataset pequeño de casas:
una lista donde cada elemento es un diccionario con la información de una casa.

In [18]:
# Dataset de casas — simula nuestra "base de conocimiento privada"
# En RAG real, estos datos vendrían de una base de datos o documentos
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

<a id='2-2'></a>
### 2.2 Creando el prompt aumentado

El primer paso es diseñar un "layout" — una forma de convertir
los datos de las casas en texto legible para el modelo.

¿Por qué necesitamos esto? Porque el LLM solo entiende texto plano,
no diccionarios de Python. Necesitamos transformar los datos a un
formato que el modelo pueda leer y entender fácilmente.

In [19]:
# Función que convierte los datos de las casas en texto legible para el LLM
def house_info_layout(houses):
    layout = ''                    # empezamos con un string vacío
    for house in houses:           # iteramos sobre cada casa
        layout += (
            f"Casa ubicada en {house['address']}, {house['city']}, "
            f"{house['state']} {house['zip']} con "
            f"{house['bedrooms']} habitaciones, {house['bathrooms']} baños, "
            f"{house['square_feet']} pies cuadrados, "
            f"precio ${house['price']}, "
            f"construida en {house['year_built']}.\n"  # \n = salto de línea
        )
    return layout

# Verificamos cómo queda el texto
print(house_info_layout(house_data))

Casa ubicada en 123 Maple Street, Springfield, IL 62701 con 3 habitaciones, 2 baños, 1500 pies cuadrados, precio $230000, construida en 1998.
Casa ubicada en 456 Elm Avenue, Shelbyville, TN 37160 con 4 habitaciones, 3 baños, 2500 pies cuadrados, precio $320000, construida en 2005.



In [20]:
# Función que arma el prompt completo: datos + pregunta del usuario
def generate_prompt(query, houses):
    # Convertimos las casas a texto usando la función anterior
    houses_layout = house_info_layout(houses)

    # Armamos el prompt completo combinando los datos y la pregunta
    # Las triple comillas """ permiten escribir texto en múltiples líneas
    PROMPT = f"""
Usa la siguiente información de casas para responder la consulta del usuario.
{houses_layout}
Consulta: {query}
    """
    return PROMPT

# Verificamos cómo queda el prompt completo
print(generate_prompt("¿Cuál es la casa más cara?", houses=house_data))


Usa la siguiente información de casas para responder la consulta del usuario.
Casa ubicada en 123 Maple Street, Springfield, IL 62701 con 3 habitaciones, 2 baños, 1500 pies cuadrados, precio $230000, construida en 1998.
Casa ubicada en 456 Elm Avenue, Shelbyville, TN 37160 con 4 habitaciones, 3 baños, 2500 pies cuadrados, precio $320000, construida en 2005.

Consulta: ¿Cuál es la casa más cara?
    


In [21]:
query = "¿Cuál es la casa más cara? ¿Y la más grande?"

# SIN datos — solo le pasamos la pregunta directamente
respuesta_sin_datos = generate_with_single_input(
    prompt=query
)

# CON datos — le pasamos el prompt aumentado con la información de las casas
prompt_aumentado = generate_prompt(query, houses=house_data)
respuesta_con_datos = generate_with_single_input(
    prompt=prompt_aumentado
)

In [22]:
# Respuesta SIN datos
print("=== SIN datos ===")
print(respuesta_sin_datos['content'])

=== SIN datos ===
La casa más cara del mundo es la "Antilia", una lujosa residencia ubicada en Mumbai, India. Propiedad del magnate Mukesh Ambani, se estima que su valor supera los 2,000 millones de dólares. La casa cuenta con 27 pisos, helipuerto, cines, una piscina y otras comodidades opulentas.

En cuanto a la casa más grande del mundo, generalmente se menciona al "Biltmore Estate" en Carolina del Norte, Estados Unidos. Esta mansión tiene más de 16,000 metros cuadrados y fue construida en el siglo XIX por George Washington Vanderbilt II. No obstante, si hablamos de la casa más grande en términos de superficie, algunos consideran el "Penfield House" en Massachusetts, que tiene más de 20,000 metros cuadrados.

Cabe mencionar que estas clasificaciones pueden variar con el tiempo y nuevos desarrollos en el mercado inmobiliario.


In [23]:
# Respuesta CON datos
print("=== CON datos ===")
print(respuesta_con_datos['content'])

=== CON datos ===
La casa más cara es la ubicada en **456 Elm Avenue, Shelbyville, TN 37160**, con un precio de **$320,000**. 

La casa más grande es también la de **456 Elm Avenue**, con un tamaño de **2500 pies cuadrados**. 

Ambas características corresponden a la misma casa.
